In [1]:
# Install the essential libraries
%%capture
!pip install openai
!pip install gradio
!pip install chromadb

In [2]:
# Import the libraries used within this application
import os
import json
import time
import re
import requests
import gradio as gr
import chromadb
import openai
from openai import OpenAI


In [3]:
# Initialize OpenAI, and embed the API_KEY
from google.colab import userdata
api_key_value = userdata.get('API_KEY')

os.environ["OPENAI_API_KEY"] = api_key_value

# Initialize the OpenAI client
client = OpenAI(api_key=api_key_value)
print("OpenAI client initialized.")

OpenAI client initialized.


In [4]:
#Use a chroma DB to function as a RAG, to handle persistance
chroma_client = chromadb.PersistentClient(path="./chroma_db")
#collection = chroma_client.create_collection("research_memory")
collection = chroma_client.get_or_create_collection("research_memory")

In [5]:
# This function, extracts keywords from the user's query which is then used to search through the RAG and the web
def build_search_query(goal):
    prompt = f"""
Convert this into a concise academic search query.

Rules:
- 3–5 keywords only
- NO commas
- NO special characters
- NO filler words (like "proposal", "study", etc.)

Goal: {goal}
"""

    res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    return res.choices[0].message.content.strip()

In [6]:
# This function stores the research papers in the RAG
def store_papers_in_rag(papers):
    for i, p in enumerate(papers):
        if not isinstance(p, dict):
            continue

        doc = (p.get("title", "") + " " + (p.get("abstract") or "")).strip()
        if not doc:
            continue

        collection.add(
            documents=[doc],
            metadatas=[{"title": p.get("title", "No title")}],
            ids=[f"{p.get('title','')}_{i}"]
        )

In [7]:
# Search the RAG for known research papers
def search_rag(query, top_k=5):
    results = collection.query(
        query_texts=[query],
        n_results=top_k
    )

    docs = results.get("documents", [[]])[0]
    metas = results.get("metadatas", [[]])[0]

    if not docs:
        return []

    papers = []
    for doc, meta in zip(docs, metas):
        papers.append({
            "title": meta.get("title", "No title"),
            "abstract": doc
        })

    return papers

In [8]:
# Check for research papers online through semantic scholar API
def fetch_semantic_scholar(query, max_results=5, retries=3):
    url = "https://api.semanticscholar.org/graph/v1/paper/search"

    headers = {
        "Accept": "application/json",
        "User-Agent": "Mozilla/5.0"
    }

    params = {
        "query": query,
        "limit": max_results,
        "fields": "title,abstract,year"
    }

    for attempt in range(retries):
        try:
            time.sleep(1)
            response = requests.get(url, headers=headers, params=params, timeout=10)

            if response.status_code == 200:
                return response.json().get("data", [])

            elif response.status_code == 429:
                print(f"Rate limited (attempt {attempt+1}). Retrying...")
                time.sleep(2)

            else:
                print(f"API Error: {response.status_code}")
                return []

        except Exception as e:
            print(f"Error: {e}")
            return []

    print("Failed after retries.")
    return []

In [9]:
# PLANNER
def create_plan(goal):
    prompt = f"""
Break the following goal into 3-4 concrete steps.

Return ONLY JSON:
[
  {{"id": 1, "task": "...", "status": "pending"}}
]

Goal: {goal}
"""

    res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )

    try:
        tasks = json.loads(res.choices[0].message.content)
    except:
        # fallback plan
        tasks = [
            {"id": 1, "task": "Search related research papers", "status": "pending"},
            {"id": 2, "task": "Analyze novelty", "status": "pending"},
            {"id": 3, "task": "Suggest improvements", "status": "pending"}
        ]

    return tasks

In [10]:
# ACTION DECIDER
def decide_action(task):
    t = task.lower()

    if "search" in t:
        return "search"
    elif "analy" in t:
        return "analyze"
    elif "improve" in t or "suggest" in t:
        return "generate"
    else:
        return "analyze"

In [11]:
# Decides if the papers in the RAG are relevant to the user's query or not
def is_relevant(papers, query):
    query_words = set(query.lower().split())

    relevant = []
    for p in papers:
        title = p.get("title", "").lower()

        if any(word in title for word in query_words):
            relevant.append(p)

    return relevant

In [12]:

#The tool used to execute the needed tasks
def execute_tool(action, task, state):
    goal = state["goal"]

    # SEARCH (RAG + FILTER + API + FALLBACK)

    if action == "search":
        query = build_search_query(goal)

        print(f"\nSearch Query: {query}")

        # 1️) Try ChromaDB first
        papers = search_rag(query)

        # 2) Relevance filter (Check if the query matches the research papers in the RAG
        def filter_relevant(papers, query):
            query_words = set(query.lower().split())
            relevant = []

            for p in papers:
                if not isinstance(p, dict):
                    continue

                title = p.get("title", "").lower()

                if any(word in title for word in query_words):
                    relevant.append(p)

            return relevant

        papers = filter_relevant(papers, query)

        if papers:
            print(f"Found {len(papers)} relevant papers in RAG")

        else:
            print("No relevant results in RAG, querying semantic scholar...")

            raw_papers = fetch_semantic_scholar(query)

            # Clean API results
            papers = []
            for p in raw_papers:
                if isinstance(p, dict) and "title" in p:
                    papers.append(p)

            # 3) Fallback if API fails
            if not papers:
                print("Semantic Scholar failed. Using fallback knowledge...")

                fallback_prompt = f"""
List 5 well-known research papers related to:

{query}

Return only paper titles (one per line).
"""

                res = client.chat.completions.create(
                    model="gpt-4o",
                    messages=[{"role": "user", "content": fallback_prompt}],
                    temperature=0
                )

                import re
                titles = res.choices[0].message.content.split("\n")

                papers = []
                for t in titles:
                    t = t.strip()

                    # Remove numbering and quotes
                    t = re.sub(r"^\d+\.\s*", "", t)

                    t = t.strip('"')

                    if t:
                        papers.append({"title": t, "abstract": ""})

            # Store in RAG for reuse
            if papers:
                store_papers_in_rag(papers)

        print(f"Papers Found: {len(papers)}")

        if papers:
            print("Top Results:")
            for p in papers[:3]:
                print(f"- {p.get('title', 'No title')}")

        state["papers"] = papers

        if not papers:
            return "No valid papers found."

        return "\n".join([p.get("title", "No title") for p in papers[:5]])

    # ANALYZE
    elif action == "analyze":
        papers = state.get("papers", [])

        if not papers:
            context = "No papers found. Use general knowledge."
        else:
            context_list = []
            for p in papers:
                if not isinstance(p, dict):
                    continue

                title = p.get("title", "No title")
                abstract = p.get("abstract") or ""

                context_list.append(f"{title}: {abstract[:200]}")

            context = "\n".join(context_list)

        print(f"Using {len(papers)} papers for analysis")

        prompt = f"""
Evaluate novelty of this idea:

IDEA:
{goal}

RELATED WORK:
{context}

Return:
- Novelty score (1-10)
- Short reasoning
"""

        res = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = res.choices[0].message.content.strip()
        state["analysis"] = output

        return output

    # GENERATE (IMPROVEMENTS)
    elif action == "generate":
        analysis = state.get("analysis", "")

        prompt = f"""
Based on this analysis:

{analysis}

Suggest 3 concrete ways to improve the novelty of this idea:

{goal}
"""
        res = client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )

        output = res.choices[0].message.content.strip()
        state["ideas"] = output

        return output

    # FALLBACK
    return "Invalid action"

In [13]:
# AGENT LOOP / Can be simplified using tools such as Langchain
def run_agent(goal):
    print(f"\nGoal: {goal}\n")

    # PLAN
    tasks = create_plan(goal)

    state = {
        "goal": goal,
        "tasks": tasks,
        "papers": [],
        "analysis": ""
    }

    print("PLAN:")
    for t in tasks:
        print(f"- {t['task']}")
    print("\n" + "="*50)

    # EXECUTION LOOP
    for task in state["tasks"]:
        print(f"\nTASK {task['id']}: {task['task']}")

        action = decide_action(task["task"])
        print(f"ACTION: {action}")

        result = execute_tool(action, task["task"], state)

        task["status"] = "completed"

        print(f"RESULT:\n{result[:500]}")
        print("-"*50)

    # FINAL SUMMARY
    print("\nFINAL OUTPUT:\n")

    summary_prompt = f"""
You are summarizing the results.

Combine the following into a clear final answer:

GOAL:
{state["goal"]}

NOVELTY ANALYSIS:
{state.get("analysis", "")}

IMPROVEMENT SUGGESTIONS:
{state.get("ideas", "")}

Requirements:
- Start with novelty score
- Briefly explain why
- Then give 3 actionable improvements
- Keep it concise and structured
"""
    res = client.chat.completions.create(
        model="gpt-4o",
        messages=[{"role": "user", "content": summary_prompt}],
        temperature=0
    )

    #print and return the final results

    print(res.choices[0].message.content)
    return res.choices[0].message.content

In [14]:
goal = """We propose using transformers for log analysis"""

In [15]:
run_agent(goal)


Goal: We propose using transformers for log analysis

PLAN:
- Search related research papers
- Analyze novelty
- Suggest improvements


TASK 1: Search related research papers
ACTION: search

Search Query: Transformers Log Analysis


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:07<00:00, 10.5MiB/s]


No relevant results in RAG, querying semantic scholar...
Rate limited (attempt 1). Retrying...
Rate limited (attempt 2). Retrying...
Papers Found: 5
Top Results:
- AI-Powered Log Analysis with BERT for Automated Event Detection and Categorization
- LRQ-DiT: Log-Rotation Post-Training Quantization of Diffusion Transformers for Image and Video Generation
- Multi Perspective Robust Deep Analysis Cyber Anomaly Detection on HDFS Log using Transformers
RESULT:
AI-Powered Log Analysis with BERT for Automated Event Detection and Categorization
LRQ-DiT: Log-Rotation Post-Training Quantization of Diffusion Transformers for Image and Video Generation
Multi Perspective Robust Deep Analysis Cyber Anomaly Detection on HDFS Log using Transformers
Log-Based Anomaly Detection with Transformers Pre-Trained on Large-Scale Unlabeled Data
Towards Log-driven Testing through Transformers: A Preliminary Study
--------------------------------------------------

TASK 2: Analyze novelty
ACTION: analyze
Using 5 p

'**Novelty Score: 3**\n\nThe concept of using transformers for log analysis is not particularly novel, as existing research has extensively explored this application in areas like anomaly detection, event detection, and software testing. While the general idea is well-established, there is potential for innovation in specific methodologies or niche applications.\n\n**Improvement Suggestions:**\n\n1. **Domain-Specific Adaptations**: Tailor transformer models for niche or underexplored domains such as healthcare, finance, or IoT, where log data has unique characteristics. Customizing the architecture or training process for these specific challenges can create a novel application.\n\n2. **Hybrid Models**: Develop hybrid approaches by combining transformers with other machine learning techniques, such as graph neural networks or unsupervised learning, to enhance performance and provide new insights. This can lead to innovative methodologies that leverage multiple techniques.\n\n3. **Real-

In [16]:
goal = """We propose using CNN for image classification"""

In [17]:
run_agent(goal)


Goal: We propose using CNN for image classification

PLAN:
- Search related research papers
- Analyze novelty
- Suggest improvements


TASK 1: Search related research papers
ACTION: search

Search Query: CNN image classification
Found 1 relevant papers in RAG
Papers Found: 1
Top Results:
- LRQ-DiT: Log-Rotation Post-Training Quantization of Diffusion Transformers for Image and Video Generation
RESULT:
LRQ-DiT: Log-Rotation Post-Training Quantization of Diffusion Transformers for Image and Video Generation
--------------------------------------------------

TASK 2: Analyze novelty
ACTION: analyze
Using 1 papers for analysis
RESULT:
Novelty Score: 2

Reasoning: The use of Convolutional Neural Networks (CNNs) for image classification is a well-established and extensively researched area in the field of computer vision and machine learning. CNNs have been a foundational technique for image classification tasks for over a decade, with numerous models and architectures developed and refined

'**Final Answer:**\n\n**Novelty Score: 2**\n\nThe use of Convolutional Neural Networks (CNNs) for image classification is a well-established and extensively researched area in computer vision and machine learning. Since the introduction of AlexNet in 2012, CNNs have become a foundational technique for image classification, resulting in a lack of novelty. Recent advancements, such as diffusion transformers and quantization, highlight the contrast with the foundational nature of CNNs.\n\n**Improvement Suggestions:**\n\n1. **Incorporate Hybrid Models:**\n   - Combine CNNs with advanced architectures like Vision Transformers (ViTs) or Graph Neural Networks (GNNs) to leverage their strengths and introduce novelty.\n\n2. **Utilize Self-Supervised Learning:**\n   - Implement self-supervised learning techniques, such as contrastive learning, to train CNNs on unlabeled data, enhancing feature learning and adding novelty.\n\n3. **Integrate Explainability and Interpretability:**\n   - Develop met

In [18]:
proposal = """Using Mixup for overfitting in Machine Learning"""

In [19]:
run_agent(proposal)


Goal: Using Mixup for overfitting in Machine Learning

PLAN:
- Search related research papers
- Analyze novelty
- Suggest improvements


TASK 1: Search related research papers
ACTION: search

Search Query: Mixup overfitting machine learning
No relevant results in RAG, querying semantic scholar...
Rate limited (attempt 1). Retrying...
Papers Found: 5
Top Results:
- Using Ultrasound Image Augmentation and Ensemble Predictions to Prevent Machine-Learning Model Overfitting
- Puzzle Mix: Exploiting Saliency and Local Statistics for Optimal Mixup
- Integrating Mixup and ArcFace for enhanced anomalous sound detection
RESULT:
Using Ultrasound Image Augmentation and Ensemble Predictions to Prevent Machine-Learning Model Overfitting
Puzzle Mix: Exploiting Saliency and Local Statistics for Optimal Mixup
Integrating Mixup and ArcFace for enhanced anomalous sound detection
Balanced-MixUp for Highly Imbalanced Medical Image Classification
UMIX: Improving Importance Weighting for Subpopulation Shift

'**Final Answer:**\n\n**Novelty Score: 4**\n\nThe use of Mixup to address overfitting in machine learning is not entirely novel, as it is a well-established data augmentation technique. Mixup has been widely applied in various contexts, such as medical imaging and sound detection, to improve model generalization and reduce overfitting. However, there is potential for novelty if a unique approach or significant improvements in a specific domain are demonstrated.\n\n**Improvement Suggestions:**\n\n1. **Domain-Specific Customization**: Develop a tailored version of Mixup optimized for niche domains like quantum computing data, financial time series, or ecological modeling. This could introduce a fresh perspective by addressing unique challenges and characteristics of these data types.\n\n2. **Integration with Advanced Techniques**: Combine Mixup with cutting-edge machine learning techniques, such as transformer models or graph neural networks, in novel ways. Exploring synergies with self-

In [20]:
proposal = """Attention is all you need, we are working on an alternative solution"""

In [21]:
run_agent(proposal)


Goal: Attention is all you need, we are working on an alternative solution

PLAN:
- Search related research papers
- Analyze novelty
- Suggest improvements


TASK 1: Search related research papers
ACTION: search

Search Query: Attention alternative solution
No relevant results in RAG, querying semantic scholar...
Rate limited (attempt 1). Retrying...
Rate limited (attempt 2). Retrying...
Rate limited (attempt 3). Retrying...
Failed after retries.
Semantic Scholar failed. Using fallback knowledge...
Papers Found: 5
Top Results:
- Efficient Attention: Attention with Linear Complexities
- Linformer: Self-Attention with Linear Complexity
- Reformer: The Efficient Transformer
RESULT:
Efficient Attention: Attention with Linear Complexities
Linformer: Self-Attention with Linear Complexity
Reformer: The Efficient Transformer
Longformer: The Long-Document Transformer
Performer: Linear Attention for Efficient Transformer Models
--------------------------------------------------

TASK 2: Analyze

'**Novelty Score: 3**\n\n**Explanation:**\nThe idea of developing an alternative to the "Attention is All You Need" model is not particularly novel, as numerous existing works like Linformer, Reformer, Longformer, and Performer already aim to enhance the original transformer model\'s efficiency and scalability. The current proposal lacks specificity and does not introduce a distinct approach that sets it apart from ongoing research trends.\n\n**Improvement Suggestions:**\n\n1. **Introduce a New Mechanism**: Develop a fundamentally different mechanism that either replaces or significantly alters the traditional attention mechanism. Consider integrating concepts from other fields, such as neuroscience or quantum computing, to create a novel approach to information processing.\n\n2. **Target Unexplored Applications**: Apply the alternative solution to domains or problems not extensively explored with transformer models, such as niche areas in natural language processing, computer vision, 

In [22]:
import gradio as gr

def app(proposal):
    try:
        result = run_agent(proposal)
        return result if result else "No output generated."
    except Exception as e:
        return f"Error: {str(e)}"


with gr.Blocks() as demo:

    gr.Markdown("# EurekaPilot")
    gr.Markdown("Evaluates and improves ML ideas using multi-step reasoning.")
    gr.Markdown(" Search →  Analyze →  Generate →  Improve")

    with gr.Row():

        # LEFT: Input
        with gr.Column():
            proposal_input = gr.Textbox(
                lines=6,
                placeholder="Enter your ML proposal here...",
                label=" Research Idea"
            )

            submit_btn = gr.Button("Submit")
            clear_btn = gr.Button("Clear")

        # RIGHT: Output (FIXED SIZE)
        with gr.Column():
            output_box = gr.Textbox(
                lines=20,
                label=" Agent Output",
                show_copy_button=True
            )

    # Button actions
    submit_btn.click(
        fn=app,
        inputs=proposal_input,
        outputs=output_box
    )

    clear_btn.click(
        fn=lambda: "",
        inputs=None,
        outputs=proposal_input
    )

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e1780e6d65832004d3.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
